In [32]:
import pandas as pd
import time
import numpy as np

In [33]:
file_path = r'avoid_recompute.csv'
df = pd.read_csv(file_path, encoding='utf-8')

In [34]:
# 放大数据量（模拟10万行）
df = pd.concat([df]*10000, ignore_index=True)
print(f"测试数据量：{len(df)} 行")

测试数据量：100000 行


In [35]:
# 计算目标：
# 1. 毛利 = (售价-成本) * 销量
# 2. 净利 = 毛利 - 毛利*税率/100 - 配送费*销量 - 毛利*折扣/100 + 毛利*退货率/100
# 3. 净利率 = 净利/毛利 * 100
# 4. 判断：净利率>10% 为"达标"，否则"不达标"

In [36]:
start_time = time.time()
# 问题：多次重复计算「(df['售价(元)']-df['成本(元)'])*df['销量']」（毛利）
df['毛利_重复'] = (df['售价(元)'] - df['成本(元)']) * df['销量']
df['净利_重复'] = (df['售价(元)'] - df['成本(元)']) * df['销量'] - \
(df['售价(元)'] - df['成本(元)']) * df['销量'] * df['税率(%)']/100 - \
                df['配送费(元)'] * df['销量'] - \
                (df['售价(元)'] - df['成本(元)']) * df['销量'] * df['促销折扣(%)']/100 + \
                (df['售价(元)'] - df['成本(元)']) * df['销量'] * df['退货率(%)']/100
df['净利率_重复'] = df['净利_重复'] / ((df['售价(元)'] - df['成本(元)']) * df['销量']) * 100
df['是否达标_重复'] = df['净利率_重复'].apply(lambda x: '达标' if x>10 else '不达标')

recompute_time = time.time() - start_time
print("\n2. 重复计算（低效方式）：")
print(f"耗时：{recompute_time:.4f} 秒")


2. 重复计算（低效方式）：
耗时：0.0215 秒


In [37]:
df

,商品ID,成本(元),售价(元),销量,税率(%),促销折扣(%),配送费(元),退货率(%),毛利_重复,净利_重复,净利率_重复,是否达标_重复
0,P001,1500,2999,20,13,5,15,2,29980,24883.20,82.999333,达标
1,P002,100,199,25,13,0,8,1,2475,1978.00,79.919192,达标
2,P003,80,159,18,13,3,6,0,1422,1086.48,76.405063,达标
3,P004,50,99,30,13,0,5,3,1470,1173.00,79.795918,达标
4,P005,180,299,22,13,2,10,1,2618,2031.48,77.596639,达标
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,8,20,0,37981,29624.99,77.999500,达标
99996,P007,90,199,28,13,0,9,2,3052,2464.28,80.743119,达标
99997,P008,40,89,35,13,1,5,4,1715,1368.50,79.795918,达标
99998,P009,1600,2999,40,13,6,18,1,55960,45167.20,80.713367,达标


In [38]:
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*10000, ignore_index=True)

start_time = time.time()
# 步骤1：预计算并缓存中间结果（毛利），后续直接复用
df['毛利_缓存'] = (df['售价(元)'] - df['成本(元)']) * df['销量']
# 步骤2：复用缓存的毛利，避免重复计算
df['净利_缓存'] = df['毛利_缓存'] - \
                df['毛利_缓存'] * df['税率(%)']/100 - \
                df['配送费(元)'] * df['销量'] - \
                df['毛利_缓存'] * df['促销折扣(%)']/100 + \
                df['毛利_缓存'] * df['退货率(%)']/100
df['净利率_缓存'] = df['净利_缓存'] / df['毛利_缓存'] * 100
df['是否达标_缓存'] = df['净利率_缓存'].apply(lambda x: '达标' if x>10 else '不达标')

cache_time = time.time() - start_time
speed_up = recompute_time / cache_time
print(f"\n3. 缓存中间结果（推荐方式）：")
print(f"耗时：{cache_time:.4f} 秒（提速 {speed_up:.1f} 倍！）")


3. 缓存中间结果（推荐方式）：
耗时：0.0211 秒（提速 1.0 倍！）


In [39]:
df

,商品ID,成本(元),售价(元),销量,税率(%),促销折扣(%),配送费(元),退货率(%),毛利_缓存,净利_缓存,净利率_缓存,是否达标_缓存
0,P001,1500,2999,20,13,5,15,2,29980,24883.20,82.999333,达标
1,P002,100,199,25,13,0,8,1,2475,1978.00,79.919192,达标
2,P003,80,159,18,13,3,6,0,1422,1086.48,76.405063,达标
3,P004,50,99,30,13,0,5,3,1470,1173.00,79.795918,达标
4,P005,180,299,22,13,2,10,1,2618,2031.48,77.596639,达标
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,8,20,0,37981,29624.99,77.999500,达标
99996,P007,90,199,28,13,0,9,2,3052,2464.28,80.743119,达标
99997,P008,40,89,35,13,1,5,4,1715,1368.50,79.795918,达标
99998,P009,1600,2999,40,13,6,18,1,55960,45167.20,80.713367,达标


In [40]:
# 重置数据
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*10000, ignore_index=True)

start_time = time.time()
# 步骤1：预计算所有高频复用的中间变量
gross_profit = (df['售价(元)'] - df['成本(元)']) * df['销量']
# 临时变量，不写入DataFrame
delivery_cost = df['配送费(元)'] * df['销量']
tax_deduct = gross_profit * df['税率(%)']/100
discount_deduct = gross_profit * df['促销折扣(%)']/100
return_add = gross_profit * df['退货率(%)']/100
# 步骤2：复用预计算变量
df['毛利_预计算'] = gross_profit
df['净利_预计算'] = gross_profit - tax_deduct - delivery_cost - discount_deduct + return_add
df['净利率_预计算'] = df['净利_预计算'] / gross_profit * 100
df['是否达标_预计算'] = np.where(df['净利率_预计算']>10, '达标', '不达标')  # 替代apply，更快

precompute_time = time.time() - start_time
speed_up_pre = recompute_time / precompute_time
print(f"\n4. 预计算多变量（复杂场景推荐）：")
print(f"耗时：{precompute_time:.4f} 秒（提速 {speed_up_pre:.1f} 倍！）")


4. 预计算多变量（复杂场景推荐）：
耗时：0.0169 秒（提速 1.3 倍！）


In [41]:
df

,商品ID,成本(元),售价(元),销量,税率(%),促销折扣(%),配送费(元),退货率(%),毛利_预计算,净利_预计算,净利率_预计算,是否达标_预计算
0,P001,1500,2999,20,13,5,15,2,29980,24883.20,82.999333,达标
1,P002,100,199,25,13,0,8,1,2475,1978.00,79.919192,达标
2,P003,80,159,18,13,3,6,0,1422,1086.48,76.405063,达标
3,P004,50,99,30,13,0,5,3,1470,1173.00,79.795918,达标
4,P005,180,299,22,13,2,10,1,2618,2031.48,77.596639,达标
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,8,20,0,37981,29624.99,77.999500,达标
99996,P007,90,199,28,13,0,9,2,3052,2464.28,80.743119,达标
99997,P008,40,89,35,13,1,5,4,1715,1368.50,79.795918,达标
99998,P009,1600,2999,40,13,6,18,1,55960,45167.20,80.713367,达标


In [42]:
# 定义带缓存的计算函数
def calc_profit(df):
    # 内部缓存中间结果
    gross = (df['售价(元)'] - df['成本(元)']) * df['销量']
    net = gross - gross*df['税率(%)']/100 - df['配送费(元)']*df['销量'] - gross*df['促销折扣(%)']/100 + gross*df['退货率(%)']/100
    rate = net/gross * 100
    df['毛利_函数'] = gross
    df['净利_函数'] = net
    df['净利率_函数'] = rate
    df['是否达标_函数'] = np.where(rate>10, '达标', '不达标')
    return df

# 重置数据
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*10000, ignore_index=True)

start_time = time.time()
df = calc_profit(df)
func_time = time.time() - start_time
speed_up_func = recompute_time / func_time
print(f"\n5. 函数封装+内部缓存：")
print(f"耗时：{func_time:.4f} 秒（提速 {speed_up_func:.1f} 倍！）")


5. 函数封装+内部缓存：
耗时：0.0169 秒（提速 1.3 倍！）


In [43]:
df

,商品ID,成本(元),售价(元),销量,税率(%),促销折扣(%),配送费(元),退货率(%),毛利_函数,净利_函数,净利率_函数,是否达标_函数
0,P001,1500,2999,20,13,5,15,2,29980,24883.20,82.999333,达标
1,P002,100,199,25,13,0,8,1,2475,1978.00,79.919192,达标
2,P003,80,159,18,13,3,6,0,1422,1086.48,76.405063,达标
3,P004,50,99,30,13,0,5,3,1470,1173.00,79.795918,达标
4,P005,180,299,22,13,2,10,1,2618,2031.48,77.596639,达标
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,8,20,0,37981,29624.99,77.999500,达标
99996,P007,90,199,28,13,0,9,2,3052,2464.28,80.743119,达标
99997,P008,40,89,35,13,1,5,4,1715,1368.50,79.795918,达标
99998,P009,1600,2999,40,13,6,18,1,55960,45167.20,80.713367,达标


In [44]:
# print("\n6. 计算结果一致性验证：")
# is_same_cache = np.isclose(df['净利率_重复'], df['净利率_缓存']).all()
# is_same_pre = np.isclose(df['净利率_重复'], df['净利率_预计算']).all()
# print(f"重复计算 vs 缓存中间结果：{is_same_cache}")
# print(f"重复计算 vs 预计算多变量：{is_same_pre}")

In [46]:
print("\n7. 高频重复计算场景：")
print("🔹 场景1：多次计算同一列组合（如 (A-B)*C）→ 缓存为中间列/变量")
print("🔹 场景2：多次调用同一过滤条件（如 df[A>100]）→ 缓存为condition变量")
print("🔹 场景3：多次apply同一复杂函数 → 封装函数+内部缓存")
print("\n优化口诀：「一次计算，多次复用；中间结果，缓存优先」")


7. 高频重复计算场景：
🔹 场景1：多次计算同一列组合（如 (A-B)*C）→ 缓存为中间列/变量
🔹 场景2：多次调用同一过滤条件（如 df[A>100]）→ 缓存为condition变量
🔹 场景3：多次apply同一复杂函数 → 封装函数+内部缓存

优化口诀：「一次计算，多次复用；中间结果，缓存优先」


In [47]:
summary = pd.DataFrame({'计算方式': ['重复计算', '缓存中间结果', '预计算多变量', '函数封装+缓存'],'耗时(秒)': [recompute_time, cache_time, precompute_time, func_time],'提速倍数': [1, speed_up, speed_up_pre, speed_up_func]}).round(2)
print("\n8. 优化效果汇总：")
print(summary)


8. 优化效果汇总：
      计算方式  耗时(秒)  提速倍数
0     重复计算   0.02  1.00
1   缓存中间结果   0.02  1.02
2   预计算多变量   0.02  1.27
3  函数封装+缓存   0.02  1.27
